In [29]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [30]:
df = pd.read_csv("../PreviousDataset.csv")

In [31]:
numeric_cols = ['SSTA', 'ClimSST', 'Depth', 'Longitude_Degrees',
                'Latitude_Degrees', 'Average_Bleaching', 'Windspeed',
                'TSA', 'TSA_DHW', 'SSTA_DHW', 'SSTA_Minimum']
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year

# Filter
df = df[df['SSTA'] > 0]
df = df[df['Year'] >= 2003]

In [32]:
df = df.sort_values(['Latitude_Degrees', 'Longitude_Degrees', 'Date'])
df['SSTA_Rolling_Mean'] = df.groupby(['Latitude_Degrees', 'Longitude_Degrees'])['SSTA'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['TSA_Rolling_Mean'] = df.groupby(['Latitude_Degrees', 'Longitude_Degrees'])['TSA'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

drop_cols = ['City_Town', 'City_Town_2', 'City_Town_3', 'Ocean', 'Realm', 'Ecoregion', 'ID', 'Country_Name', 'State_Island_Province', 'Date2', 'SSTA_Mean']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Month encoding
df['Month'] = df['Date'].dt.month
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
df = df.drop(columns=['Month', 'Date'])

# Train

In [33]:
X = df.drop("Average_Bleaching", axis=1)
Y = df["Average_Bleaching"]

In [34]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [35]:
print(X_train_raw.isna().any().any())
print(X_test_raw.isna().any().any())

True
True


In [36]:
# Impute
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)


In [37]:
scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

In [38]:
pca = PCA(n_components=0.95) # Increased variance capture
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

## OpTuna

In [39]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [40]:
def rf_objective(trial):

    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_regression,
                                 k=trial.suggest_int("k_features", 5, X.shape[1]))),
        ('pca', PCA(
            n_components=trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

rf_study = optuna.create_study(direction="maximize")

rf_study.optimize(rf_objective, n_trials=50)

print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

[I 2026-05-01 09:08:49,975] A new study created in memory with name: no-name-864b8cd2-8891-4886-a65a-93266d889c67
[I 2026-05-01 09:08:50,668] Trial 0 finished with value: 0.2358647434693603 and parameters: {'n_estimators': 67, 'max_depth': 13, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'k_features': 36, 'pca_variance': 0.9362399557387355}. Best is trial 0 with value: 0.2358647434693603.
[I 2026-05-01 09:08:51,322] Trial 1 finished with value: 0.2083700878283971 and parameters: {'n_estimators': 65, 'max_depth': 21, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'k_features': 36, 'pca_variance': 0.9892749546562314}. Best is trial 0 with value: 0.2358647434693603.
[I 2026-05-01 09:08:52,594] Trial 2 finished with value: 0.14750091653651254 and parameters: {'n_estimators': 268, 'max_depth': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'k_features': 40, 'pca_variance': 0.9372710150302491}. Best is trial 0 with value: 0.2358647434693603.
[I 2026-05-01 09:08:54,168] Trial 3 finished 

========== RF BEST ==========
{'n_estimators': 160, 'max_depth': 32, 'min_samples_leaf': 1, 'max_features': 'log2', 'k_features': 20, 'pca_variance': 0.9324003641302269}
Best R²: 0.2826


In [41]:
def xgb_objective(trial):

    model = xgb.XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(
            score_func=f_regression,
            k=trial.suggest_int("k_features", 5, X.shape[1])
        )),
        ('pca', PCA(
            n_components=trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

xgb_study = optuna.create_study(direction="maximize")

xgb_study.optimize(xgb_objective, n_trials=50)

print("========== XGB BEST ==========")
print(xgb_study.best_params)
print(f"Best R²: {xgb_study.best_value:.4f}")

[I 2026-05-01 09:10:58,359] A new study created in memory with name: no-name-d715b4f0-dba7-4e8b-bd5f-0e023859efb7
[I 2026-05-01 09:11:01,275] Trial 0 finished with value: 0.13348163487255582 and parameters: {'n_estimators': 276, 'learning_rate': 0.0994280604652042, 'max_depth': 13, 'subsample': 0.8183717675280137, 'colsample_bytree': 0.6378997623803502, 'k_features': 7, 'pca_variance': 0.9613476234998041}. Best is trial 0 with value: 0.13348163487255582.
[I 2026-05-01 09:11:06,484] Trial 1 finished with value: 0.1829384071522849 and parameters: {'n_estimators': 296, 'learning_rate': 0.09492290205019621, 'max_depth': 13, 'subsample': 0.8629143253277127, 'colsample_bytree': 0.5588163444810522, 'k_features': 19, 'pca_variance': 0.9164951396031583}. Best is trial 1 with value: 0.1829384071522849.
[I 2026-05-01 09:11:07,240] Trial 2 finished with value: 0.2636071567990751 and parameters: {'n_estimators': 261, 'learning_rate': 0.019159781026952722, 'max_depth': 6, 'subsample': 0.786815489635

========== XGB BEST ==========
{'n_estimators': 282, 'learning_rate': 0.026808106142845907, 'max_depth': 9, 'subsample': 0.5951036655520707, 'colsample_bytree': 0.7347258198885658, 'k_features': 39, 'pca_variance': 0.9240912509767348}
Best R²: 0.3122


## Start!

In [42]:
xgb_model = xgb.XGBRegressor(
    n_estimators=282, 
    learning_rate=0.026, 
    max_depth=9, 
    subsample=0.6, 
    colsample_bytree=0.7, 
    n_jobs=-1,
    random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=160, 
    max_depth=32, 
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1, 
    random_state=42
)

In [43]:
voting = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
voting.fit(X_train_pca, y_train)

# Evaluation
preds = voting.predict(X_test_pca)
r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n--- Leakage-Free Results ---")
print(f"R-squared: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


--- Leakage-Free Results ---
R-squared: 0.4456
RMSE: 7.2122


In [44]:
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# %%
# K-Fold Validation 

n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

r2_scores   = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
    y_train,     y_test     = Y.iloc[train_idx], Y.iloc[test_idx]

    # Impute
    imputer     = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    # Scale
    scaler      = StandardScaler()
    X_train_ss  = scaler.fit_transform(X_train_imp)
    X_test_ss   = scaler.transform(X_test_imp)

    # PCA
    pca         = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_ss)
    X_test_pca  = pca.transform(X_test_ss)

    # Train
    voting      = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
    voting.fit(X_train_pca, y_train)

    # Evaluate
    preds = voting.predict(X_test_pca)
    r2_scores.append(r2_score(y_test, preds))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, preds)))

    print(f"Fold {fold}  →  R²: {r2_scores[-1]:.4f}  |  RMSE: {rmse_scores[-1]:.4f}")

print(f"\n--- {n_splits}-Fold Summary ---")
print(f"R²   mean: {np.mean(r2_scores):.4f}  ±  {np.std(r2_scores):.4f}")
print(f"RMSE mean: {np.mean(rmse_scores):.4f}  ±  {np.std(rmse_scores):.4f}")

Fold 1  →  R²: 0.4153  |  RMSE: 7.1665
Fold 2  →  R²: 0.3890  |  RMSE: 8.5010
Fold 3  →  R²: 0.2182  |  RMSE: 8.6102
Fold 4  →  R²: 0.3703  |  RMSE: 6.6309

--- 4-Fold Summary ---
R²   mean: 0.3482  ±  0.0767
RMSE mean: 7.7271  ±  0.8507
